# Graph Profile of Quantum Circuits

Given an input circuit $C$ and an architecture graph $AG$, we are interested in the following graph properties
1. Is $IG(C)$ embeddable in $AG$? If not, is it complete?
2. For every segment of $k$ gates, how many qubits it involves? Is there a particular gate which appears only once? If yes, this gate can sometimes be replaced with remote CNOT gate.
3. Is there a (maximal) star? Here I mean a subgraph which has a node that connects to every other node and there are no other edges.
4. Could a subgraph be further partitioned?
5. For each edge, attach a weight or a distribution of its occurrence in the layers

In [2]:
from collections import defaultdict
import copy
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.dagcircuit import DAGCircuit, DAGOpNode, DAGInNode, DAGOutNode
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler.layout import Layout
from qiskit.circuit.quantumregister import Qubit
from qiskit.visualization import dag_drawer, circuit_drawer
from qiskit.qasm import Qasm

import matplotlib.pyplot as plt
import networkx as nx
import rustworkx as rx

from dac_part import is_rx_embeddable, is_embeddable, draw_nx_graph
#from dac_part import get_interaction_graph


def node_depth(dag: DAGCircuit):
    # Initialize a dictionary to store the depth of each node
    node_depths = defaultdict(int)

    # Perform a topological traversal of the DAG
    for node in dag.topological_op_nodes():
        # Initialize depth to 0
        depth = 0
        predecessors = list(dag.predecessors(node))

        # If there are predecessors, set depth to the maximum depth of predecessors + 1
        if predecessors:
            depth = max(node_depths[predecessor] for predecessor in predecessors) + 1

        # Store the depth of the node in the dictionary
        node_depths[node] = depth
    
    return node_depths

def get_interaction_graph(dag: DAGCircuit) -> nx.Graph:
    g = nx.Graph()
    qubit_layers = defaultdict(list)
    edge_layers = defaultdict(list)

    for layer_no, layer in enumerate(dag.layers()):
        #print(f"layer number{layer_no} {layer['partition']}")
        #if layer_no == 0: continue
        for qubit_pair in layer['partition']:
            p, q = qubit_pair[0]._index, qubit_pair[1]._index
            g.add_edge(p,q)
            qubit_layers[p].append(layer_no)
            qubit_layers[q].append(layer_no)
            edge_layers[(p,q)].append(layer_no)
            edge_layers[(q,p)].append(layer_no)

    for node, layers in qubit_layers.items():
        g.nodes[node]['layer_distribution'] = layers

    for edge, layers in edge_layers.items():
        g.edges[edge]['layer_distribution'] = layers
    return g

def graph_of_circuit(circuit):
    ''' Return the interaction graph of the reduced QuantumCircuit C
            - node set: indices of qubits in C
            - edge set: all pair (p,q) if CNOT [p,q] or CNOT[q,p] in C
        Args:
            C (list): the input reduced circuit
        Returns:
            g (Graph)
    '''   
    g = nx.Graph()
    for gate in circuit:
        if gate.operation.name != 'cx': continue
        p, q = gate.qubits[0]._index, gate.qubits[1]._index
        g.add_edge(p,q)
    return g  

def edge_distribution(dag: DAGCircuit):
    node_depths = node_depth(dag)
    EdgeDistr = defaultdict(list)
    for node in dag.nodes():
        if not isinstance(node, DAGOpNode): 
            continue
        if len(node.qargs) > 2:
            raise Exception ('dag only contains 1- or 2-qubit gates')
        if len(node.qargs) == 1:
            raise Exception ('dag is reduced and only contains non-consecutive 2-qubit gates')
        p, q = node.qargs[0]._index, node.qargs[1]._index
        d = node_depths[node]
        EdgeDistr[(p,q)].append(d)
        EdgeDistr[(q,p)].append(d)
    return EdgeDistr 
    
def layer_distribution_graph_of_dag(dag):
    qc = dag_to_circuit(dag)
    g = graph_of_circuit(qc)
    EdgeDistr = edge_distribution(dag)
    edge_weights = {}
    node_weights = {p:{'layer_distribution':[]} for p in g.nodes()}
    for edge in g.edges():
        edge_weights[edge] = EdgeDistr[(edge[0],edge[1])]
        node_weights[edge[0]]['layer_distribution'] += EdgeDistr[(edge[0],edge[1])]
        node_weights[edge[1]]['layer_distribution'] += EdgeDistr[(edge[0],edge[1])]
        #print(edge, node_weights[edge[0]], node_weights[edge[1]])
    for p in g.nodes():
        gate_list = copy.copy(node_weights[p]['layer_distribution'])
        gate_list.sort()
        #print(p, gate_list)
        node_weights[p]['layer_distribution'] = gate_list
        
    nx.set_node_attributes(g, node_weights)    
    nx.set_edge_attributes(g, edge_weights, 'layer_distribution')
    return g

In [3]:
"""Consider reduced QuantumCircuit"""
def graph_profile(qc: QuantumCircuit, printOK=False):

    #qc should be reduced first!
    redqc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)
    dag = circuit_to_dag(redqc)

    ig = get_interaction_graph(dag)
    if printOK: 
        print(f'The DAGCircuit has {dag.num_qubits()} qubits, depth {dag.depth()}, {dag.count_ops()}')
    
    Node_Layer = dict()
    if printOK: 
        print('~~~~~~~~~~~~~~~~~~~~~~~~')                                 
        print('*node distribution*')
    for node in ig.nodes():
        if printOK: 
            print(node, ig.nodes[node]['layer_distribution'])
        Node_Layer[node] = ig.nodes[node]['layer_distribution']
    if printOK: 
        print('========================')                                 
    
    Edge_Layer = dict()
    if printOK:
        print('~~~~~~~~~~~~~~~~~~~~~~~~')                                 
        print('*edge distribution*')
    for edge in ig.edges():
        if printOK:
            print(edge, ig.edges[edge]['layer_distribution'])
        Edge_Layer[edge] = ig.edges[edge]['layer_distribution']
        
    if printOK: 
        print('========================')                                
        # Draw the graph
        pos = nx.spring_layout(ig)  # Define the layout (you can choose other layouts)
        nx.draw(ig, pos, with_labels=True, node_size=500, node_color='skyblue', font_size=20)
        plt.show()  # Display the graph
    
    return ig, Node_Layer, Edge_Layer

def Layer_Edge(ig, Edge_Layer, depth, is_convex=False):
    """return the edges/cxs which occupy this layer"""
    LE = dict()
    for layer_no in range(depth):
        if not is_convex:
            edge_list = [edge for edge in ig.edges() if layer_no in Edge_Layer[edge]]
        else:
            edge_list = [edge for edge in ig.edges() if min(Edge_Layer[edge]) <= layer_no <= max(Edge_Layer[edge])]
        LE[layer_no] = edge_list
    return LE
        

def cutting_point(i, selected_edges, Edge_Layer, ig):
    """layer i a cutting point w.r.t. edges in EX 
            if for each edge e out of EX, its distribution is either all before i or all after i """
        
    for edge in ig.edges():
        #print(f"cutting point test for layer {i} and edge {edge} with {Edge_Layer[edge]}")
        if edge in selected_edges: 
            continue
        if len(Edge_Layer[edge])>1 and min(Edge_Layer[edge]) < i < max(Edge_Layer[edge]):
            return False
    return True

def all_cutting_points(selected_edges, Edge_Layer, depth, ig):
    
    cutting_list = [i for i in range(depth) if cutting_point(i, selected_edges, Edge_Layer, ig)]
    return cutting_list

In [4]:
"""Calculate the cutting points and compute the graph of each section"""
def dynamic_graph_partition(qc, lev=1):
    newcirc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)
    dag = circuit_to_dag(newcirc)
    print(f'orginal circuit info: {dag.count_ops(), dag.depth()}')
    g, _, Edge_Layer = graph_profile(newcirc, True)

    if lev != 1:
        raise Exception ('We currently only consider the level 1 case.')
        
    selected_edges = []
    cutting_list = all_cutting_points(selected_edges, Edge_Layer, dag.depth(), g)
    cutting_list.sort()
    print(f'The cutting points are {cutting_list}')
    
    if not cutting_list:
        return [g]
    
    """show the graph of each section"""
    GRAPHS = []
    for idx in range(len(cutting_list)+1):
        "determine the section corresponding to each idx"
        if idx == 0: #the first section
            LayerList = list(range(cutting_list[0]+1))
        elif idx == len(cutting_list): #the last section
            LayerList = list(range(cutting_list[-1], dag.depth()))
            #print(f'the end {cutting_list[-1], LayerList}')            
        else:
            LayerList = list(range(cutting_list[idx-1]+1, cutting_list[idx]+1))
            
        print(f"section {idx}: {LayerList}")                     
        edgelist = [edge for edge in g.edges() if set(Edge_Layer[edge])&set(LayerList)]
        newgraph = nx.Graph()
        newgraph.add_edges_from(edgelist)
        draw_nx_graph(newgraph)
        GRAPHS.append(newgraph)           
    return GRAPHS

In [6]:
import matplotlib.pyplot as plt
import networkx as nx
from dac_part import remove_1q_and_consecutive_2q_gates_in_circuit

from ag import qgrid, q20
import time
import os

path = '../bench/qiskit_circuit_benchmark/' 
#filename = 'excitation_preserving_6.qasm'
filename = 'grover_operator_6.qasm'
#filename = 'quantum_volume_16.qasm'
#filename = 'phase_oracle_14.qasm'
#filename = 'qft_6.qasm'
#filename = 'phase_estimation_6.qasm'

print(filename)
#AG = q20()
AG = qgrid(2,3)
rxAG = rx.networkx_converter(AG)

with open(path+filename, 'r') as file:
    qasm_code = file.read()

# Create a QuantumCircuit from the QASM code
qc = QuantumCircuit.from_qasm_str(qasm_code)
print(qc.count_ops())

newcirc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)

dag = circuit_to_dag(newcirc)
print(filename, dag.count_ops(), dag.depth())

g, _, Edge_Layer = graph_profile(newcirc, True)
max_layer_num = dag.depth()
selected_edges = []
cutting_list = all_cutting_points(selected_edges, Edge_Layer, max_layer_num, g)
print(f'The cutting points are {cutting_list}')
selected_edges = [(3,4)]
cutting_list = all_cutting_points(selected_edges, Edge_Layer, max_layer_num, g)
print(f'The cutting points for {selected_edges} are {cutting_list}')

dynamic_graph_partition(qc, lev=1)

grover_operator_6.qasm
OrderedDict([('u1', 93), ('cx', 92), ('u', 27), ('u3', 2)])
grover_operator_6.qasm {'cx': 61} 61
The DAGCircuit has 6 qubits, depth 61, {'cx': 61}
~~~~~~~~~~~~~~~~~~~~~~~~
*node distribution*
4 [0, 1, 3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59]
5 [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60]
3 [1, 2, 3, 4, 5, 9, 17, 25, 33, 41, 49, 57]
2 [5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 37, 53]
1 [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 45]
0 [29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
~~~~~~~~~~~~~~~~~~~~~~~~
*edge distribution*
(4, 5) [0]
(4, 3) [1, 3]
(4, 2) [7, 11]
(4, 1) [15, 19, 23, 27]
(4, 0) [31, 35, 39, 43, 47, 51, 55, 59]
(5, 3) [2, 4]
(5, 2) [6, 8, 10, 12]
(5, 1) [14, 16, 18, 20, 22, 24, 26, 28]
(5, 0) [30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60]
(3

TypeError: '_AxesStack' object is not callable

<Figure size 640x480 with 0 Axes>

# circuit separation
Consider the following example. It is clear that the circuit can be decomposed into two parts at layer 29. From layer 30, all gates involve qubit 0.

or_6.qasm {'cx': 61} 61
layer distribution of (4, 5) is (1, 1, 1)
layer distribution of (4, 3) is (2, 2, 4)
layer distribution of (4, 2) is (2, 8, 12)
layer distribution of (4, 1) is (4, 16, 28)
layer distribution of (4, 0) is (8, 32, 60)
layer distribution of (5, 3) is (2, 3, 5)
layer distribution of (5, 2) is (4, 7, 13)
layer distribution of (5, 1) is (8, 15, 29)
layer distribution of (5, 0) is (16, 31, 61)
layer distribution of (3, 2) is (2, 6, 10)
layer distribution of (3, 1) is (2, 18, 26)
layer distribution of (3, 0) is (4, 34, 58)
layer distribution of (2, 1) is (2, 14, 22)
layer distribution of (2, 0) is (2, 38, 54)
layer distribution of (1, 0) is (2, 30, 46)

In [64]:
# Weighted SUBGRAPH initial mapping
#TODO: We should also calculate where to insert the mapping

def wtggraph(dag: DAGCircuit, AG: nx.Graph): 
    ''' Return a graph g which is isomorphic to a subgraph of AG
            while maximizing the number of CNOTs in C that correspond to edges in g
        Method: sort the edges according to their weights (the number of CNOTs in C corresponding to each edge);
                construct a graph by starting with the edge with the largest weight; then consider the edge with the second large weight, ...
                if in any step the graph is not isomorphic to a subgraph of G, skip this edge and consider the next till all edges are considered.
    '''    
    g_of_c = layer_distribution_graph_of_dag(dag)
    test = is_embeddable(g_of_c, AG, 10)
    print(f'The graph of the circuit is embeddable in G? {test[0]}')
    if test[0]:
        print('The graph of the circuit is embeddable in G')
        return g_of_c, test[1]
    
    for edge in g_of_c.edges():
        print(edge, nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])    
    
    edge_wgt_list = list([len(nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge]), edge] for edge in g_of_c.edges())
    edge_wgt_list.sort(key=lambda t: t[0], reverse=True) # q[0] weight, q[1] edge'
    #print(edge_wgt_list)

    
    '''Sort the edges reversely according to their weights''' 
    EdgeList = list(item[1] for item in edge_wgt_list)    
    #edge_num = len(EdgeList)
    
    '''We search backward, remove the first edge that makes g not embeddable, 
            and continue till all edges are evaluated in sequence. '''
            
    g = nx.Graph()
    result = dict()
    
    # add the first edge into g
    edge = EdgeList[0]
    g.add_edge(edge[0], edge[1])
    num_gate_sat = len(nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])
    
    for edge in EdgeList:           
        g.add_edge(edge[0], edge[1])           
        test = is_embeddable(g, AG, 10)
        if not test[0]:
            g.remove_edge(edge[0], edge[1])
            if nx.degree(g, edge[0]) == 0: g.remove_node(edge[0])
            if nx.degree(g, edge[1]) == 0: g.remove_node(edge[1])
        else:
            result = test[1]
            num_gate_sat += len(nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])
    
    #! either (p,q) or (q,p) in nx.get_edge_attributes(g_of_c, 'layer_distribution')
    for edge in g.edges():
        if edge not in nx.get_edge_attributes(g_of_c, 'layer_distribution'):
            edge = (edge[1],edge[0])
        print(edge, nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])
    return g, result, num_gate_sat

In [36]:
gx, result, num_sat = wtggraph(dag, AG)
print(gx.nodes(), result, num_sat)

The graph of the circuit is embeddable in G? False
(3, 4) [1, 3, 6, 8, 11, 13, 16, 18, 21, 23, 26, 28, 31, 33]
(3, 0) [5]
(3, 1) [10, 15]
(3, 2) [20, 25, 30, 35]
(4, 0) [2, 4]
(4, 1) [7, 9, 12, 14]
(4, 2) [17, 19, 22, 24, 27, 29, 32, 34]
(0, 2) [37]
(0, 1) [38]
(1, 2) [36]
(3, 4) [1, 3, 6, 8, 11, 13, 16, 18, 21, 23, 26, 28, 31, 33]
(3, 0) [5]
(4, 2) [17, 19, 22, 24, 27, 29, 32, 34]
(4, 1) [7, 9, 12, 14]
(0, 2) [37]
[3, 4, 2, 1, 0] {4: 3, 2: 2, 0: 4, 3: 5, 1: 1} 42
